##1. Instalações

In [0]:
%pip install python-dotenv

%pip install azure-storage-file-datalake azure-identity python-dotenv

%restart_python


## 2. Imports

In [0]:
from dotenv import load_dotenv
import os
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

import pandas as pd
import io

## 3. Carregamento das Variáveis de Ambiente

In [0]:
load_dotenv()

## 4. Leitura e Validação das Variáveis

In [0]:
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account_name = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
container_name = os.getenv("ADLS_CONTAINER_NAME")

## 5. Carregamento do .env:

In [0]:
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv("/Workspace/Users/soaress.elias@gmail.com/merca-data-platform-categorias/.env")

CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT_NAME = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
CONTAINER_NAME = os.getenv("ADLS_CONTAINER_NAME")

## 6. Conexão com o Data Lake

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

account_url = f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"

service_client = DataLakeServiceClient(
    account_url=account_url,
    credential=credential
)

file_system_client = service_client.get_file_system_client(CONTAINER_NAME)

##7. Análise exploratória de dados

In [0]:
# 1 - Descobrir tipos de arquivo no Data Lake:
paths = file_system_client.get_paths()

for path in paths:
    print(path.name)

In [0]:
# 2 — Obter metadados dos arquivos:
for path in file_system_client.get_paths():
    print(
        f"Nome: {path.name}",
        f"Tipo: {'Diretório' if path.is_directory else 'Arquivo'}",
        sep="\n"
    )
    print("-" * 50)

In [0]:
# 3 - Análise exploratória de CSVs - Data Quality:

# ============================================================
# Função de análise exploratória voltada para Data Quality
# ============================================================

def analisar_csv(caminho_arquivo):
    nome_arquivo = caminho_arquivo.split("/")[-1]

    print("\n" + "#" * 80)
    print(f"📂 INICIANDO ANÁLISE: {nome_arquivo}")
    print("#" * 80)

    try:
        # ----------------------------------------------------
        # Leitura do arquivo
        # ----------------------------------------------------

        print("\n📥 Leitura do arquivo...")

        file_client = file_system_client.get_file_client(
            caminho_arquivo
        )

        download = file_client.download_file()
        conteudo = download.readall()

        df = pd.read_csv(io.BytesIO(conteudo))

        print("✅ Arquivo carregado com sucesso!")

        # ----------------------------------------------------
        # Visão geral
        # ----------------------------------------------------

        print("\n" + "=" * 70)
        print("📊 VISÃO GERAL DO DATASET")
        print("=" * 70)

        linhas, colunas = df.shape

        print(f"• Dataset               : {nome_arquivo}")
        print(f"• Quantidade de linhas : {linhas:,}")
        print(f"• Quantidade de colunas: {colunas}")
        print(f"• Total de células     : {linhas * colunas:,}")

        print("\n🔎 Primeiras 5 linhas:")
        print(df.head())

        print("\n📋 Informações gerais:")
        df.info()

        # ----------------------------------------------------
        # Completude
        # ----------------------------------------------------

        print("\n" + "=" * 70)
        print("🧩 COMPLETUDE DOS DADOS")
        print("=" * 70)

        nulos = df.isnull().sum()
        percentual_nulos = (
            df.isnull().mean() * 100
        ).round(2)

        resumo_nulos = pd.DataFrame({
            "Qtd. Nulos": nulos,
            "% Nulos": percentual_nulos
        })

        resumo_nulos = resumo_nulos.sort_values(
            by="% Nulos",
            ascending=False
        )

        print(resumo_nulos)

        total_nulos = nulos.sum()

        if total_nulos == 0:
            print("\n✅ Não foram encontrados valores nulos.")
        else:
            print(
                f"\n⚠️ Total de valores nulos encontrados: "
                f"{total_nulos:,}"
            )

        # ----------------------------------------------------
        # Duplicidade
        # ----------------------------------------------------

        print("\n" + "=" * 70)
        print("🔁 ANÁLISE DE DUPLICIDADE")
        print("=" * 70)

        duplicados = df.duplicated().sum()

        print(
            f"Registros duplicados: {duplicados:,}"
        )

        if duplicados == 0:
            print("✅ Nenhum registro duplicado encontrado.")
        else:
            percentual_dup = (
                duplicados / len(df)
            ) * 100

            print(
                f"⚠️ {percentual_dup:.2f}% dos registros "
                "estão duplicados."
            )

        # ----------------------------------------------------
        # Tipos de dados
        # ----------------------------------------------------

        print("\n" + "=" * 70)
        print("🧱 TIPOS DE DADOS")
        print("=" * 70)

        tipos = pd.DataFrame({
            "Tipo de dado": df.dtypes.astype(str)
        })

        print(tipos)

        # ----------------------------------------------------
        # Cardinalidade
        # ----------------------------------------------------

        print("\n" + "=" * 70)
        print("🔢 CARDINALIDADE")
        print("=" * 70)

        cardinalidade = pd.DataFrame({
            "Valores únicos": df.nunique()
        })

        cardinalidade = cardinalidade.sort_values(
            by="Valores únicos",
            ascending=False
        )

        print(cardinalidade)

        print(
            "\n💡 Cardinalidade alta pode indicar "
            "identificadores únicos."
        )

        # ----------------------------------------------------
        # Estatísticas descritivas
        # ----------------------------------------------------

        print("\n" + "=" * 70)
        print("📈 ESTATÍSTICAS DESCRITIVAS")
        print("=" * 70)

        print(df.describe())

        print(
            "\n💡 Verifique possíveis outliers, "
            "valores mínimos/máximos inesperados "
            "e dispersões elevadas."
        )

        # ----------------------------------------------------
        # Resumo executivo
        # ----------------------------------------------------

        print("\n" + "=" * 70)
        print("📝 RESUMO EXECUTIVO")
        print("=" * 70)

        print(f"Dataset               : {nome_arquivo}")
        print(f"Linhas                : {linhas:,}")
        print(f"Colunas               : {colunas}")
        print(f"Valores nulos         : {total_nulos:,}")
        print(f"Registros duplicados  : {duplicados:,}")

        print("\n✅ Análise concluída com sucesso.")

    except Exception as e:
        print(
            f"\n❌ Erro ao analisar "
            f"{nome_arquivo}: {e}"
        )

# ============================================================
# Lista de arquivos a serem analisados
# ============================================================

arquivos_csv = [
    "batch-data/ecommerce_categorias.csv",
    "batch-data/ecommerce_clientes.csv",
    "batch-data/ecommerce_enderecos.csv",
    "batch-data/ecommerce_itens_pedido.csv",
    "batch-data/ecommerce_pedidos.csv",
    "batch-data/ecommerce_produtos.csv",
    "batch-data/ecommerce_rastreamento_entregas.csv",
    "batch-data/physical_itens_venda_caixa.csv",
    "batch-data/physical_lojas.csv",
    "batch-data/physical_vendas_caixa.csv"
]

# ============================================================
# Execução das análises
# ============================================================

for arquivo in arquivos_csv:
    analisar_csv(arquivo)

In [0]:
# 4 - Análise exploratória de arquivos parquet - Data Quality:

#Mini-framework automatizado de Data Quality para arquivos Parquet

# ============================================================
# Função de análise exploratória para arquivos Parquet
# ============================================================

def analisar_parquet(caminho_arquivo):

    nome_arquivo = caminho_arquivo.split("/")[-1]
    pasta_lote = caminho_arquivo.split("/")[-2]

    print("\n" + "#" * 90)
    print(f"📂 INICIANDO ANÁLISE: {nome_arquivo}")
    print(f"🕒 LOTE/TIMESTAMP: {pasta_lote}")
    print("#" * 90)

    try:

        # ====================================================
        # Leitura do arquivo
        # ====================================================

        print("\n📥 Leitura do arquivo parquet...")

        file_client = file_system_client.get_file_client(
            caminho_arquivo
        )

        download = file_client.download_file()

        conteudo = download.readall()

        tamanho_mb = len(conteudo) / (1024 ** 2)

        df = pd.read_parquet(
            io.BytesIO(conteudo),
            engine="pyarrow"
        )

        print("✅ Arquivo carregado com sucesso.")
        print(f"📦 Tamanho do arquivo : {tamanho_mb:.2f} MB")
        print("⚙️ Engine utilizada   : pyarrow")

        # ====================================================
        # Visão geral
        # ====================================================

        print("\n" + "=" * 70)
        print("📊 VISÃO GERAL DO DATASET")
        print("=" * 70)

        linhas, colunas = df.shape

        print(f"Dataset               : {nome_arquivo}")
        print(f"Lote                  : {pasta_lote}")
        print(f"Quantidade de linhas  : {linhas:,}")
        print(f"Quantidade de colunas : {colunas}")
        print(f"Total de células      : {linhas * colunas:,}")

        print("\n🔎 Primeiras 5 linhas:")
        print(df.head())

        print("\n📋 Informações gerais:")
        df.info()

        # ====================================================
        # Completude
        # ====================================================

        print("\n" + "=" * 70)
        print("🧩 COMPLETUDE DOS DADOS")
        print("=" * 70)

        nulos = df.isnull().sum()

        percentual_nulos = (
            df.isnull().mean() * 100
        ).round(2)

        resumo_nulos = pd.DataFrame({
            "Qtd. Nulos": nulos,
            "% Nulos": percentual_nulos
        })

        resumo_nulos = resumo_nulos.sort_values(
            by="% Nulos",
            ascending=False
        )

        print(resumo_nulos)

        total_nulos = nulos.sum()

        if total_nulos == 0:
            print("\n✅ Não foram encontrados valores nulos.")
        else:
            print(
                f"\n⚠️ Total de valores nulos: "
                f"{total_nulos:,}"
            )

        # ====================================================
        # Duplicidade
        # ====================================================

        print("\n" + "=" * 70)
        print("🔁 ANÁLISE DE DUPLICIDADE")
        print("=" * 70)

        duplicados = df.duplicated().sum()

        print(
            f"Registros duplicados: {duplicados:,}"
        )

        if duplicados == 0:
            print("✅ Nenhum registro duplicado encontrado.")
        else:

            percentual_dup = (
                duplicados / len(df)
            ) * 100

            print(
                f"⚠️ {percentual_dup:.2f}% dos registros "
                "estão duplicados."
            )

        # ====================================================
        # Tipos de dados
        # ====================================================

        print("\n" + "=" * 70)
        print("🧱 TIPOS DE DADOS")
        print("=" * 70)

        tipos = pd.DataFrame({
            "Tipo de dado": df.dtypes.astype(str)
        })

        print(tipos)

        # ====================================================
        # Cardinalidade
        # ====================================================

        print("\n" + "=" * 70)
        print("🔢 CARDINALIDADE")
        print("=" * 70)

        cardinalidade = pd.DataFrame({
            "Valores únicos": df.nunique()
        })

        cardinalidade = cardinalidade.sort_values(
            by="Valores únicos",
            ascending=False
        )

        print(cardinalidade)

        print(
            "\n💡 Cardinalidade elevada pode indicar "
            "identificadores únicos."
        )

        # ====================================================
        # Estatísticas numéricas
        # ====================================================

        print("\n" + "=" * 70)
        print("📈 ESTATÍSTICAS DESCRITIVAS")
        print("=" * 70)

        print(df.describe())

        # ====================================================
        # Estatísticas para colunas categóricas
        # ====================================================

        print("\n" + "=" * 70)
        print("🏷️ COLUNAS CATEGÓRICAS")
        print("=" * 70)

        print(
            df.describe(
                include=["object", "category"]
            )
        )

        # ====================================================
        # Datas
        # ====================================================

        colunas_datetime = (
            df.select_dtypes(
                include=["datetime"]
            )
            .columns
            .tolist()
        )

        print("\n" + "=" * 70)
        print("📅 COLUNAS DE DATA")
        print("=" * 70)

        if len(colunas_datetime) == 0:

            print(
                "ℹ️ Nenhuma coluna datetime encontrada."
            )

        else:

            for coluna in colunas_datetime:

                print(f"\nColuna: {coluna}")

                print(
                    f"Data mínima: "
                    f"{df[coluna].min()}"
                )

                print(
                    f"Data máxima: "
                    f"{df[coluna].max()}"
                )

        # ====================================================
        # Resumo executivo
        # ====================================================

        print("\n" + "=" * 70)
        print("📝 RESUMO EXECUTIVO")
        print("=" * 70)

        print(f"Dataset               : {nome_arquivo}")
        print(f"Lote                  : {pasta_lote}")
        print(f"Tamanho (MB)          : {tamanho_mb:.2f}")
        print(f"Linhas                : {linhas:,}")
        print(f"Colunas               : {colunas}")
        print(f"Valores nulos         : {total_nulos:,}")
        print(f"Duplicados            : {duplicados:,}")

        print("\n✅ Análise concluída com sucesso.")

    except Exception as e:

        print(
            f"\n❌ Erro ao analisar "
            f"{nome_arquivo}: {e}"
        )

# Execução automática para todos os arquivos Parquet:
timestamps = [
    "101144",
    "101202",
    "101402",
    "101602",
    "101801",
    "102002",
    "102202",
    "102602",
    "102801",
    "103001",
    "103402",
    "103802",
    "104002",
    "104202",
    "104402",
    "104602",
    "104802",
    "105202",
    "105402",
    "105602",
    "110002",
    "110402",
    "110602",
    "110801",
    "111402",
    "111602",
    "112001",
    "112201",
    "112402",
    "112602",
    "112802",
    "113002",
    "113202",
    "113402",
    "113802"
]

arquivos = [
    "ecommerce_categorias.parquet",
    "ecommerce_clientes.parquet",
    "ecommerce_enderecos.parquet",
    "ecommerce_itens_pedido.parquet",
    "ecommerce_pedidos.parquet",
    "ecommerce_produtos.parquet",
    "ecommerce_rastreamento.parquet"
]

caminhos_parquet = []

for timestamp in timestamps:
    for arquivo in arquivos:

        caminhos_parquet.append(
            f"real-time-data/2026/06/06/"
            f"{timestamp}/{arquivo}"
        )

# Execução do framework:

for caminho in caminhos_parquet:
    analisar_parquet(caminho)

##8. Criar da tabela e processo de ingestão:

In [0]:
# 1 - Direcionar file_system_client para o container squad1:
file_system_client2 = service_client.get_file_system_client("squad1")

In [0]:
# 2 — Carga inicial (CSV):

# Ler CSV:
csv_client = file_system_client.get_file_client(
    "batch-data/ecommerce_categorias.csv"
)

csv_bytes = csv_client.download_file().readall()

import io
import pandas as pd

df = pd.read_csv(io.BytesIO(csv_bytes))

In [0]:
# Verificar leitura:
df.head()

df.info()

df.isnull().sum()

In [0]:
# 3 — Salvar no staging:

# Convertendo para parquet:
buffer = io.BytesIO()

df.to_parquet(
    buffer,
    index=False
)

buffer.seek(0)

In [0]:
# Salvando na pasta staging:
file_client = file_system_client2.get_file_client(
    "staging/ecommerce_categorias/carga_inicial.parquet"
)

file_client.upload_data(
    buffer.getvalue(),
    overwrite=True
)

In [0]:
# Conferir destino:
paths = file_system_client2.get_paths()

for path in paths:
    print(path.name)

In [0]:
# 3 — Ingestão incremental (arquivos parquet - tempo real):

# Ler arquivo controle.txt:
import io
import pandas as pd

try:
    controle_client = file_system_client2.get_file_client(
        "staging/ecommerce_categorias/controle.txt"
    )

    conteudo = (
        controle_client
        .download_file()
        .readall()
        .decode("utf-8")
    )

    arquivos_processados = set(conteudo.splitlines())

except:
    arquivos_processados = set()

print(f"{len(arquivos_processados)} arquivos já processados.")

In [0]:
# Encontrar os novos arquivos parquet:
paths = file_system_client.get_paths(path="real-time-data")

parquets = []

for p in paths:
    if p.name.endswith("ecommerce_categorias.parquet"):
        parquets.append(p.name)

parquets = sorted(parquets)

print(f"{len(parquets)} arquivos encontrados.")

In [0]:
# Processar os novos arquivos:
for arquivo in parquets:

    if arquivo in arquivos_processados:
        continue

    print("Processando:", arquivo)

    dados = (
        file_system_client
        .get_file_client(arquivo)
        .download_file()
        .readall()
    )

    df_inc = pd.read_parquet(io.BytesIO(dados))

    timestamp = arquivo.split("/")[-2]

    buffer = io.BytesIO()

    df_inc.to_parquet(buffer, index=False)

    buffer.seek(0)

    destino = (
        f"staging/ecommerce_categorias/"
        f"lote_{timestamp}.parquet"
    )

    file_system_client2.get_file_client(destino).upload_data(
        buffer.getvalue(),
        overwrite=True
    )

    arquivos_processados.add(arquivo)

In [0]:
# Atualizar o controle:
conteudo = "\n".join(sorted(arquivos_processados))

controle_client = file_system_client2.get_file_client(
    "staging/ecommerce_categorias/controle.txt"
)

controle_client.upload_data(
    conteudo.encode("utf-8"),
    overwrite=True
)

print("Controle atualizado.")

In [0]:
# Conferir destino:
paths = file_system_client2.get_paths()

for path in paths:
    print(path.name)